In [3]:
# List all the files in a folder
import math
import os
import random
from typing import List, Tuple

from bargaining_table import (split_into_blocks, 
                              game_signature, 
                              player_files, 
                              sample_pairs,
                              extract_svo_angle
                              )

In [4]:
num_samples = 352  # Number of random pairs to sample
sample_with_replacement = False  # Whether to sample with replacement
folder_path_player1 = "./data/Dor-humans/stage-2-analysis/Number2players/"
folder_path_player2 = "./data/Dor-humans/stage-2-analysis/Number1players/"

# Retrieve all the games played by both players
player1_files = player_files(folder_path_player1, "PlayerNmber2")
player2_files = player_files(folder_path_player2, "PlayerNmber1")

sampled_pairs = sample_pairs(
    player1_files,
    player2_files,
    num_samples,
    with_replacement=sample_with_replacement,
)

In [5]:
# Filter pairs that match the same game signature
matching_pairs = []
for f1, f2 in sampled_pairs:
    sign_f1 = game_signature(os.path.join(folder_path_player1, f1))
    sign_f2 = game_signature(os.path.join(folder_path_player2, f2))
    if sign_f1 == sign_f2:
        matching_pairs.append((f1, f2))
        
print(f"Total sampled pairs: {len(sampled_pairs)}")
print(f"Matching pairs with same game signature: {len(matching_pairs)}")

Total sampled pairs: 352
Matching pairs with same game signature: 352


In [6]:
# For each block, extract the 6 features from the first element in each pair
# Features:
# (X1): FPP (or FPC, in the data)
# (X2): Percentage of points that the player receives if both players implement a focal point
# solution according to the closeness rule.
# (X3): Dominance
# (X4): (Accumulate) the total number of points in the game
# (X5): Number of disks
# (X6): SVO angle of the other player
# Y (label): whether the players chose a focal point (1) or not (0).
games = {
    "x1": [],
    "x2": [],
    "x3": [],
    "x4": [],
    "x5": [],
    "x6": [],
    "y": [],
}

# Adaptive Agent Features FOR AGENT 1, SO WE MODEL AGENT 2!
for f1, f2 in matching_pairs:

    # Get blocks of data for both players
    blocks_p1 = split_into_blocks(os.path.join(folder_path_player1, f1))[1:]  # Skip header
    blocks_p2 = split_into_blocks(os.path.join(folder_path_player2, f2))[1:]  # Skip header

    for block, block_other_player in zip(blocks_p2, blocks_p1):

        # (X5): Number of disks
        x5 = (len(block) - 3)  # Exclude first 3 rows
        
        total_number_of_points = 0
        for (i, row), other_player_row in zip(enumerate(block), block_other_player):
            row_values = row.split(',')
            if i == 0:
            # (X1): FPP (or FPC, in the data) and
            # (X3): Dominance
                x1 = float(row_values[1])
                x3 = int(row_values[3])
                games["x1"].append(x1)
                games["x3"].append(x3)
            elif i == 1:
                # Player 1 position
                p1x, p1y = int(row_values[-2]), int(row_values[-1])
            elif i == 2:
                # Player 2 position
                p2x, p2y = int(row_values[-2]), int(row_values[-1])
            # Any row corresponding to a disk selection
            else:
                # Disk position, value, and assignment
                dx, dy = int(row_values[-4]), int(row_values[-3])
                disk_value = int(row_values[-2])
                
                # (X2): Percentage of points that the player receives if both players implement a focal point solution according to the closeness rule.
                # !! CHANGE THIS BLOCK FOR PLAYER 2
                distance_p1_disk = math.sqrt((p1x - dx) ** 2 + (p1y - dy) ** 2)
                distance_p2_disk = math.sqrt((p2x - dx) ** 2 + (p2y - dy) ** 2)
                decision_p1 = ('r2' if distance_p1_disk < distance_p2_disk else 'r1')
                decision_p2 = ('r1' if distance_p2_disk < distance_p1_disk else 'r2')
                if decision_p1 == decision_p2:
                    x2 = (1. if decision_p1 == 'r2' else 0)  # normalised
                else:
                    x2 = 0.2  # normalised: it would be (0.2 * disk_value / disk_value)
                games["x2"].append(x2)
                
                # (X4): (Accumulate) the total number of points
                total_number_of_points += disk_value
                
                # Compute the label (Y): whether the players chose a focal point
                other_player_values = other_player_row.split(',')
                assignment_player1 = row_values[-1]
                assignment_other_player = other_player_values[-1]
                if assignment_player1 == assignment_other_player:
                    games["y"].append(1)
                else:
                    games["y"].append(0)
                
        
        games["x1"].extend([x1]*(x5-1)) # Repeat for each disk in the block (minus one, already present)
        games["x3"].extend([x3]*(x5-1))
        games["x4"].extend([total_number_of_points]*x5)  # Repeat for each disk in the block
        games["x5"].extend([x5]*x5)  # Repeat for each disk in the block
        # games["x6"].extend([total_number_of_points]*x5)
        
        # (X6): SVO angle of the player
        # Now it gets messed up because of the inconsistent namings
        f1_gamenum = f1.split('BT')[1].split('_PlayerNmber2.csv')[0]
        try:
            svo_angle = extract_svo_angle(os.path.join(folder_path_player1, f"SVO{f1_gamenum}_L.csv"))
        except FileNotFoundError:
            try:
                svo_angle = extract_svo_angle(os.path.join(folder_path_player1, f"SVO{f1_gamenum}_R.csv"))
            except FileNotFoundError:
                svo_angle = 22.5  # Default value if SVO file not found    
        games["x6"].extend([svo_angle]*x5)  # Repeat for each disk in the block

In [7]:
import pandas as pd

# Save to a DataFrame
df = pd.DataFrame(games)

# Save to file
df.to_csv("./data/Dor-humans/bargaining_games_player_blue.csv", index=False)


In [ ]:
import joblib
import numpy as np

MODEL_PATH = "./logs/bargaining_table_rf.joblib"
model = joblib.load(MODEL_PATH)

# Replace the values below with the unseen game's features (x1..x6)
unseen_game = np.array([[1.7246, 1.0, 0, 12, 5, -7.8152924]])

probabilities = model.predict_proba(unseen_game)
label = (probabilities[:, 1] >= 0.5).astype(int)  # 50% threshold

print("Probabilities:", probabilities)
print("Predicted label:", label[0])